In [ ]:
import os, subprocess, numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from matplotlib.gridspec import GridSpec
import matplotlib.patheffects as pe

BASE = os.path.expanduser('~/chipseq_tnfr2')
os.makedirs(BASE, exist_ok=True)

BAM_DIR  = f'{BASE}/bam'
BW_DIR   = f'{BASE}/bigwig'
PEAK_DIR = f'{BASE}/peaks'
FIG_DIR  = f'{BASE}/figures'
DIFF_DIR = f'{BASE}/diffbind_results'
INDEX    = f'{BASE}/ref/bowtie2_mm10/mm10'

for d in [BAM_DIR, BW_DIR, PEAK_DIR, FIG_DIR, DIFF_DIR]:
    os.makedirs(d, exist_ok=True)

SAMPLES = {
    'SRR29084979': {'group': 'Treg_BATF',  'type': 'ChIP'},
    'SRR29084978': {'group': 'Treg_BATF',  'type': 'ChIP'},
    'SRR29084975': {'group': 'Treg_Input', 'type': 'Input'},
    'SRR29084977': {'group': 'Tconv_BATF', 'type': 'ChIP'},
    'SRR29084976': {'group': 'Tconv_BATF', 'type': 'ChIP'},
    'SRR29084974': {'group': 'Tconv_Input','type': 'Input'},
}

print(f'✅ 工作目录: {BASE}')
print(f'✅ 子目录: bam, bigwig, peaks, figures, diffbind_results')


In [ ]:
os.chdir(BASE)
fastq_files = sorted([f for f in os.listdir('.') if f.endswith('.fastq.gz')])
print(f'找到 {len(fastq_files)} 个 FASTQ 文件:')
for f in fastq_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {f}  ({size_mb:.1f} MB)')

if len(fastq_files) == 6:
    print('\n✅ 6 个 FASTQ 文件齐全！')


In [ ]:
ref_dir = f'{BASE}/ref/bowtie2_mm10'
os.makedirs(ref_dir, exist_ok=True)
index_files = [f for f in os.listdir(ref_dir) if f.startswith('mm10') and f.endswith('.bt2')]

if len(index_files) >= 6:
    print(f'✅ mm10 索引已存在，共 {len(index_files)} 个文件')
else:
    print('⬇️ 在终端运行：')
    print(f'  cd {ref_dir}')
    print('  wget -c ftp://ftp.ccb.jhu.edu/pub/data/bowtie2_indexes/mm10.zip')
    print('  unzip mm10.zip')


In [ ]:
print('在终端运行：\n')
for srr in SAMPLES:
    bam = f'{BAM_DIR}/{srr}.sorted.bam'
    if os.path.exists(bam) and os.path.exists(bam + '.bai'):
        print(f'# ✅ {srr} 已完成')
    else:
        print(f'# {srr} ({SAMPLES[srr]["group"]})')
        print(f'bowtie2 -p 8 --very-sensitive -x {INDEX} -U {srr}.fastq.gz | samtools sort -@ 4 -o {bam} -')
        print(f'samtools index {bam}\n')

print('💡 或用 for 循环一次跑：')
print('''for SRR in SRR29084974 SRR29084975 SRR29084976 SRR29084977 SRR29084978 SRR29084979; do
  bowtie2 -p 8 --very-sensitive -x ~/chipseq_tnfr2/ref/bowtie2_mm10/mm10 \\
    -U ${SRR}.fastq.gz | samtools sort -@ 4 -o bam/${SRR}.sorted.bam -
  samtools index bam/${SRR}.sorted.bam
done''')


In [ ]:
print(f'{"SRR":<14} {"Group":<14} {"Total reads":>12} {"Mapped":>8}')
print('-'*52)
for srr, info in SAMPLES.items():
    bam = f'{BAM_DIR}/{srr}.sorted.bam'
    if not os.path.exists(bam):
        print(f'{srr:<14} {info["group"]:<14} {"N/A":>12} {"N/A":>8}')
        continue
    r = subprocess.run(['samtools', 'flagstat', bam], capture_output=True, text=True)
    lines = r.stdout.strip().split('\n')
    total = int(lines[0].split()[0])
    mapped_pct = [l for l in lines if 'mapped' in l and 'mate' not in l][0].split('(')[1].split('%')[0]
    status = '✅' if float(mapped_pct) > 70 else '⚠️'
    print(f'{srr:<14} {info["group"]:<14} {total:>12,} {mapped_pct:>7}% {status}')


In [ ]:
#!/usr/bin/env python3

from pathlib import Path
import numpy as np
import pyBigWig
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

BASE = Path(".")
WINDOW_BED = BASE / "ref" / "Tnfrsf1b.mm10.window.bed"
GENE_BED = BASE / "ref" / "Tnfrsf1b.mm10.bed"

TRACKS = [
    ("BATF Treg ChIP",  BASE / "bigwig" / "BATF_Treg.bw"),
    ("BATF Tconv ChIP", BASE / "bigwig" / "BATF_Tconv.bw"),
    ("Input Treg",      BASE / "bigwig" / "Input_Treg.bw"),
    ("Input Tconv",     BASE / "bigwig" / "Input_Tconv.bw"),
]

PEAKS = [
    ("BATF Treg peaks",  BASE / "peaks" / "BATF_Treg_peaks.narrowPeak"),
    ("BATF Tconv peaks", BASE / "peaks" / "BATF_Tconv_peaks.narrowPeak"),
]


def read_bed(path):
    fields = path.read_text().strip().splitlines()[0].split("\t")
    chrom = fields[0]
    start = int(fields[1])
    end = int(fields[2])
    name = fields[3] if len(fields) > 3 else ""
    strand = fields[4] if len(fields) > 4 else "."
    return chrom, start, end, name, strand


def same_chrom(a, b):
    return a == b or a.replace("chr", "") == b.replace("chr", "")


def choose_chrom_for_bw(bw, chrom):
    chroms = bw.chroms()
    if chrom in chroms:
        return chrom
    alt = chrom.replace("chr", "") if chrom.startswith("chr") else "chr" + chrom
    if alt in chroms:
        return alt
    raise ValueError(f"{chrom} not found in bigWig. Available examples: {list(chroms)[:5]}")


def binned_signal(bw_path, chrom, start, end, bins=800):
    with pyBigWig.open(str(bw_path)) as bw:
        bw_chrom = choose_chrom_for_bw(bw, chrom)
        edges = np.linspace(start, end, bins + 1).astype(int)
        vals = []
        for i in range(bins):
            s = int(edges[i])
            e = int(edges[i + 1])
            if e <= s:
                e = s + 1
            v = bw.stats(bw_chrom, s, e, type="mean")[0]
            vals.append(0 if v is None or np.isnan(v) else v)
        x = (edges[:-1] + edges[1:]) / 2
        return x, np.array(vals)


def load_peaks(peak_path, chrom, start, end):
    out = []
    if not peak_path.exists():
        return out
    with peak_path.open() as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            fields = line.rstrip("\n").split("\t")
            p_chrom = fields[0]
            p_start = int(fields[1])
            p_end = int(fields[2])
            if same_chrom(p_chrom, chrom) and p_end > start and p_start < end:
                out.append((max(p_start, start), min(p_end, end)))
    return out


chrom, start, end, _, _ = read_bed(WINDOW_BED)
g_chrom, g_start, g_end, g_name, g_strand = read_bed(GENE_BED)

existing_tracks = [(label, path) for label, path in TRACKS if path.exists()]
if not existing_tracks:
    raise FileNotFoundError("No bigWig files found in bigwig/. Please check your file names.")

nrows = len(existing_tracks) + 2
fig, axes = plt.subplots(
    nrows,
    1,
    figsize=(12, 1.45 * nrows),
    sharex=True,
    gridspec_kw={"height_ratios": [1] * len(existing_tracks) + [0.55, 0.55]},
)

# bigWig signal tracks
all_max = 0
signals = []
for label, path in existing_tracks:
    x, y = binned_signal(path, chrom, start, end)
    signals.append((label, x, y))
    if len(y):
        all_max = max(all_max, float(np.nanmax(y)))

for ax, (label, x, y) in zip(axes[:len(existing_tracks)], signals):
    ax.fill_between(x, y, 0)
    ax.set_ylabel(label, rotation=0, ha="right", va="center")
    ax.set_ylim(0, all_max * 1.08 if all_max > 0 else 1)
    ax.spines[["top", "right"]].set_visible(False)

# peak bars
peak_ax = axes[len(existing_tracks)]
peak_ax.set_ylim(0, len(PEAKS))
peak_ax.set_yticks([])
peak_ax.set_ylabel("MACS2\npeaks", rotation=0, ha="right", va="center")
peak_ax.spines[["top", "right", "left"]].set_visible(False)

for i, (label, peak_path) in enumerate(PEAKS):
    peaks = load_peaks(peak_path, chrom, start, end)
    y = len(PEAKS) - i - 0.65
    peak_ax.text(start, y + 0.08, label, ha="left", va="bottom", fontsize=9)
    for ps, pe in peaks:
        peak_ax.add_patch(Rectangle((ps, y), pe - ps, 0.22))

# gene model
gene_ax = axes[len(existing_tracks) + 1]
gene_ax.set_ylim(0, 1)
gene_ax.set_yticks([])
gene_ax.set_ylabel("Gene", rotation=0, ha="right", va="center")
gene_ax.spines[["top", "right", "left"]].set_visible(False)

gene_ax.plot([g_start, g_end], [0.5, 0.5], linewidth=2)

if g_strand == "-":
    gene_ax.annotate("", xy=(g_start, 0.5), xytext=(g_end, 0.5),
                     arrowprops=dict(arrowstyle="->", linewidth=2))
    tss = g_end
else:
    gene_ax.annotate("", xy=(g_end, 0.5), xytext=(g_start, 0.5),
                     arrowprops=dict(arrowstyle="->", linewidth=2))
    tss = g_start

gene_ax.axvline(tss, linestyle="--", linewidth=1)
gene_ax.text((g_start + g_end) / 2, 0.68, f"{g_name} ({g_strand})", ha="center", va="bottom")

axes[-1].set_xlim(start, end)
axes[-1].set_xlabel(f"{chrom}:{start:,}-{end:,}  mm10 / GRCm38")

fig.suptitle("BATF ChIP-seq signal at Tnfrsf1b / TNFR2 locus", y=0.995)
fig.tight_layout()

out_pdf = BASE / "plots" / "Tnfrsf1b_BATF_chipseq_peakplot.pdf"
out_png = BASE / "plots" / "Tnfrsf1b_BATF_chipseq_peakplot.png"

fig.savefig(out_pdf)
fig.savefig(out_png, dpi=300)

print(f"Saved: {out_pdf}")
print(f"Saved: {out_png}")